# 05. 폭염 분석 (12번 ②)

**질문:** 폭염 날에는 소비가 줄어드나, 아니면 낮에서 저녁으로 옮겨 가나? 어느 업종·연령·시간대가 가장 크게 반응하나?

- 자료: 카드 데이터1 (시간대 4구간, 성별·연령 있음, **거주지 없음 → 외지인 구분 없이 전체 결제**)
- 법인 결제와 제외 업종 8개(데이터2의 7개 + 통신요금(PC통신,무선호출))를 빼서 83개 업종, 7그룹
- 기간: 2025-07-01 ~ 09-30 (폭염일이 7~9월에만 있음), 연휴(8/15~17)는 뺌
- 코드: `src/heat.py` · 상태: **팀 검증 전**

## 이 셀이 하는 일: 실행 위치와 환경 확인

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':   # notebooks 폴더에서 실행된 경우
    os.chdir('..')                                   # 프로젝트 폴더로 이동
sys.path.insert(0, 'src')

import numpy, pandas, statsmodels
print('실행 폴더:', os.path.basename(os.getcwd()))
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| statsmodels', statsmodels.__version__)

실행 폴더: bigcontest_team
Python 3.14.7 | numpy 2.5.2 | pandas 3.0.5 | statsmodels 0.15.0


## 분석 방법 (쉬운 설명)

**모형:** 시간대마다 따로 `ln(결제건수) ~ 폭염 + 비 + 요일 + 공휴일 + 월` 을 추정합니다.
- 비 변수: 10~30mm, 30mm 이상, 호우 뒤 1~5일 (비 효과가 폭염 효과에 섞이지 않게)
- 표준오차: HAC(maxlags=3), 호우 분석과 같음

**폭염 정의 3가지**
| 이름 | 뜻 |
|---|---|
| hot | 최고기온 33℃ 이상 (주 정의) |
| hot_mid / hot_big | 33~35℃ / 35℃ 이상 (강도 비교) |
| warn1 / warn2 | 폭염주의보 / 폭염경보 발효일 (기상특보 CSV) |

**표 읽는 법:** 각 칸은 `점추정 [5%, 95%]` 이고 단위는 %입니다. 폭염 날이 평소보다 몇 % 많거나(+) 적은지(−) 뜻합니다.
90% 구간이 0을 포함하면 "차이가 있다고 말하기 어렵다"고 읽습니다.

**시간대 비중 변화:** `ln(그 시간대 건수 / 하루 건수)` 를 같은 모형으로 추정합니다. 낮 비중이 줄고 저녁 비중이 늘면 "소비가 저녁으로 옮겨 갔다"는 뜻입니다.

## 이 셀이 하는 일: 데이터 읽기 (카드 데이터1·2, 기상, 달력, 약 30초)

In [2]:
import pandas as pd
from event_study import load_all
from heat import (load_card1, warning_levels, effect_table, share_table, drop_one_run, show)

data = load_all()
card1 = load_card1()
pd.set_option('display.width', 250)
print('카드 데이터1 (개인, 83개 업종):', len(card1), '행')

카드 데이터1 (개인, 83개 업종): 914405 행


## 이 셀이 하는 일: 폭염특보 발효일 확인 (춘천, 강남=서울동남권 가정)

In [3]:
for reg in ['춘천', '강남']:
    lv, acts = warning_levels(reg)
    print(f'[{reg}] 주의보 {int((lv == 1).sum())}일, 경보 {int((lv == 2).sum())}일, 특보 기록 {len(acts)}건')
    print('   발효일:', ' '.join(f"{d:%m/%d}{'경' if v == 2 else '주'}" for d, v in lv[lv > 0].items()))

[춘천] 주의보 21일, 경보 19일, 특보 기록 12건
   발효일: 07/07경 07/08경 07/09경 07/10경 07/11경 07/12경 07/13경 07/21주 07/22주 07/23주 07/24경 07/25경 07/26경 07/27경 07/28경 07/29경 07/30경 07/31경 08/01경 08/02경 08/03경 08/04주 08/05주 08/06주 08/15주 08/16주 08/17주 08/18주 08/19주 08/20주 08/21주 08/22주 08/23주 08/24주 08/25경 08/27주 08/28주 08/29주 08/30주 08/31주


[강남] 주의보 21일, 경보 19일, 특보 기록 13건
   발효일: 07/07경 07/08경 07/09경 07/10경 07/11경 07/12경 07/13경 07/21주 07/22주 07/23주 07/24경 07/25경 07/26경 07/27경 07/28경 07/29경 07/30경 07/31경 08/01경 08/02경 08/03경 08/04주 08/05주 08/06주 08/15주 08/16주 08/17주 08/18주 08/19주 08/20주 08/21주 08/22주 08/23주 08/24주 08/25경 08/27주 08/28주 08/29주 08/30주 08/31주


## 춘천 결과 1. 시간대별 폭염 효과 (hot, 33℃ 이상)

In [4]:
show(effect_table(data, card1, '춘천', 'hot'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,hot,"2.5 [ 0.4, 4.6]",89,"-0.1 [ -3.5, 3.4]","2.2 [ -0.1, 4.6]","2.1 [ -0.0, 4.4]","3.3 [ 0.3, 6.4]"


## 춘천 결과 2. 강도 비교 (33~35℃ vs 35℃ 이상)

In [5]:
show(effect_table(data, card1, '춘천', 'dose'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,hot_mid,"0.7 [ -0.9, 2.3]",89,"-2.2 [ -5.8, 1.4]","0.2 [ -1.5, 1.8]","0.6 [ -1.1, 2.4]","1.3 [ -1.4, 4.1]"
1,전체,hot_big,"5.3 [ 2.3, 8.5]",89,"3.5 [ 0.2, 6.9]","5.6 [ 2.2, 9.1]","4.6 [ 1.2, 8.0]","6.6 [ 2.5, 10.8]"


## 춘천 결과 3. 폭염특보 기준 (주의보 / 경보)

In [6]:
show(effect_table(data, card1, '춘천', 'warn'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,warn1,"-0.4 [ -2.1, 1.4]",83,"-0.3 [ -4.0, 3.6]","-0.2 [ -2.7, 2.4]","-1.2 [ -3.3, 1.0]","0.5 [ -2.7, 3.9]"
1,전체,warn2,"1.7 [ -1.7, 5.2]",83,"3.0 [ -2.0, 8.2]","2.3 [ -0.7, 5.3]","-0.1 [ -3.9, 3.8]","3.8 [ -1.6, 9.4]"


## 춘천 결과 4. 시간대 비중 변화 (hot)

In [7]:
pd.DataFrame([share_table(data, card1, '춘천')]).pipe(show)

,지역,00_05,06_11,12_17,18_23
0,춘천,"-2.5 [ -5.4, 0.5]","-0.2 [ -2.0, 1.6]","-0.3 [ -1.0, 0.3]","0.8 [ -0.7, 2.4]"


## 춘천 결과 5. 업종 그룹별 (hot)

In [8]:
show(effect_table(data, card1, '춘천', 'hot', by='group'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,식료품,hot,"1.2 [ -0.6, 3.0]",89,"-0.4 [ -3.8, 3.1]","1.2 [ -0.8, 3.3]","-0.1 [ -2.1, 2.0]","2.9 [ -0.2, 6.1]"
1,외식,hot,"4.3 [ 1.1, 7.6]",89,"5.1 [ 0.5, 10.0]","4.2 [ 0.8, 7.7]","4.5 [ 1.4, 7.8]","4.0 [ 0.0, 8.1]"
2,의료·약국,hot,"-0.2 [ -2.4, 2.0]",89,"35.8 [ 1.8, 81.1]","0.4 [ -2.6, 3.4]","-1.5 [ -4.2, 1.4]","-0.3 [ -3.8, 3.4]"
3,교통·숙박,hot,"5.6 [ 3.1, 8.2]",89,"-3.4 [ -7.9, 1.4]","3.8 [ 0.5, 7.2]","6.1 [ 3.4, 8.8]","7.7 [ 5.0, 10.6]"
4,여가,hot,"3.5 [ -1.5, 8.8]",89,"-1.5 [ -6.5, 3.7]","8.6 [ 1.5, 16.3]","5.9 [ -1.5, 13.9]","-0.5 [ -4.4, 3.4]"
5,상품쇼핑,hot,"3.9 [ -1.7, 9.8]",89,"-1.5 [-34.9, 49.2]","2.2 [ -4.1, 8.8]","5.0 [ -1.0, 11.4]","4.7 [ -3.9, 14.1]"
6,생활·교육서비스,hot,"3.8 [ -0.8, 8.7]",89,"45.6 [ 4.4, 103.0]","2.5 [ -2.8, 8.2]","2.1 [ -2.9, 7.4]","10.9 [ 3.3, 19.1]"


## 춘천 결과 6. 연령별 (hot)

In [9]:
show(effect_table(data, card1, '춘천', 'hot', by='band'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,10~30대,hot,"3.2 [ 0.4, 6.2]",89,"0.6 [ -3.3, 4.7]","3.3 [ -0.3, 7.1]","3.0 [ -0.0, 6.1]","3.7 [ 0.2, 7.4]"
1,40~50대,hot,"2.9 [ 0.7, 5.1]",89,"-0.8 [ -4.1, 2.6]","2.4 [ 0.0, 4.9]","3.3 [ 0.9, 5.8]","2.8 [ -0.1, 5.7]"
2,60대 이상,hot,"-0.3 [ -1.7, 1.0]",89,"-3.9 [ -8.7, 1.2]","0.5 [ -1.4, 2.5]","-2.2 [ -4.0, -0.3]","3.7 [ 0.6, 7.0]"


## 춘천 결과 7. 폭염 묶음 하나씩 빼고 다시 추정 (하루 전체)

In [10]:
drop_one_run(data, card1, '춘천').round(1)

,뺀 묶음,일수,효과%,5%,95%
0,07/01~07/01,1,2.7,0.5,4.8
1,07/07~07/12,6,2.7,0.3,5.1
2,07/22~08/02,12,-0.2,-2.1,1.8
3,08/04~08/05,2,2.5,0.3,4.7
4,08/12~08/12,1,2.4,0.2,4.6
5,08/16~08/16,1,2.5,0.4,4.6
6,08/18~08/18,1,2.5,0.4,4.7
7,08/20~08/22,3,2.7,0.3,5.0
8,08/24~08/25,2,3.1,1.0,5.2
9,08/29~08/29,1,2.5,0.3,4.8


## 강남 결과 1. 시간대별 폭염 효과 (hot, 33℃ 이상)

In [11]:
show(effect_table(data, card1, '강남', 'hot'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,hot,"1.1 [ -0.0, 2.2]",89,"-0.1 [ -3.6, 3.4]","1.1 [ -0.0, 2.3]","0.5 [ -0.7, 1.6]","2.2 [ 0.4, 4.2]"


## 강남 결과 2. 강도 비교 (33~35℃ vs 35℃ 이상)

In [12]:
show(effect_table(data, card1, '강남', 'dose'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,hot_mid,"0.5 [ -0.7, 1.7]",89,"-1.4 [ -4.7, 1.9]","0.3 [ -0.9, 1.5]","0.4 [ -1.1, 1.9]","1.1 [ -0.7, 2.8]"
1,전체,hot_big,"1.8 [ 0.5, 3.2]",89,"1.5 [ -3.9, 7.1]","2.1 [ 0.7, 3.6]","0.5 [ -0.8, 1.9]","3.7 [ 1.2, 6.3]"


## 강남 결과 3. 폭염특보 기준 (주의보 / 경보)

In [13]:
show(effect_table(data, card1, '강남', 'warn'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,전체,warn1,"2.4 [ 1.0, 3.8]",83,"1.7 [ -1.7, 5.1]","3.6 [ 1.7, 5.6]","1.3 [ -0.2, 2.8]","3.2 [ 0.9, 5.6]"
1,전체,warn2,"3.0 [ 1.3, 4.8]",83,"2.8 [ -0.6, 6.4]","3.5 [ 1.7, 5.4]","2.1 [ 0.0, 4.2]","4.4 [ 1.1, 7.8]"


## 강남 결과 4. 시간대 비중 변화 (hot)

In [14]:
pd.DataFrame([share_table(data, card1, '강남')]).pipe(show)

,지역,00_05,06_11,12_17,18_23
0,강남,"-1.2 [ -4.1, 1.8]","0.0 [ -1.1, 1.2]","-0.6 [ -1.2, -0.1]","1.1 [ 0.0, 2.3]"


## 강남 결과 5. 업종 그룹별 (hot)

In [15]:
show(effect_table(data, card1, '강남', 'hot', by='group'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,식료품,hot,"0.6 [ -0.7, 2.0]",89,"1.6 [ -0.6, 3.8]","0.6 [ -0.7, 1.9]","-1.5 [ -2.8, -0.3]","3.8 [ 1.0, 6.6]"
1,외식,hot,"2.9 [ 1.5, 4.3]",89,"0.6 [ -2.1, 3.3]","2.9 [ 1.3, 4.6]","2.6 [ 1.2, 4.0]","3.7 [ 1.9, 5.5]"
2,의료·약국,hot,"-0.3 [ -1.7, 1.2]",89,"1.3 [ -6.0, 9.2]","0.7 [ -1.3, 2.8]","-1.4 [ -2.9, 0.1]","2.2 [ 0.0, 4.4]"
3,교통·숙박,hot,"1.1 [ -0.6, 2.9]",89,"0.5 [ -3.0, 4.0]","1.9 [ -1.4, 5.2]","1.2 [ -0.7, 3.1]","0.9 [ -1.3, 3.2]"
4,여가,hot,"2.8 [ -1.0, 6.8]",89,"4.9 [ 1.5, 8.3]","-2.9 [ -9.9, 4.6]","1.4 [ -3.1, 6.2]","4.8 [ 0.2, 9.5]"
5,상품쇼핑,hot,"0.5 [ -2.0, 3.2]",89,"-0.8 [ -5.1, 3.8]","0.7 [ -2.7, 4.2]","2.0 [ -0.4, 4.4]","-0.6 [ -3.4, 2.2]"
6,생활·교육서비스,hot,"-1.0 [ -5.6, 4.0]",89,"0.3 [-11.1, 13.2]","-1.4 [ -9.5, 7.4]","0.5 [ -5.4, 6.9]","-1.3 [ -7.3, 5.1]"


## 강남 결과 6. 연령별 (hot)

In [16]:
show(effect_table(data, card1, '강남', 'hot', by='band'))

,대상,변수,하루,표본일수,00_05,06_11,12_17,18_23
0,10~30대,hot,"1.1 [ -0.1, 2.3]",89,"-0.6 [ -4.3, 3.2]","1.3 [ -0.1, 2.7]","0.8 [ -0.3, 1.9]","1.7 [ 0.0, 3.4]"
1,40~50대,hot,"1.6 [ 0.4, 2.9]",89,"0.2 [ -3.6, 4.0]","1.5 [ 0.3, 2.7]","1.3 [ -0.2, 2.7]","2.6 [ 0.6, 4.7]"
2,60대 이상,hot,"-1.0 [ -2.0, 0.0]",89,"1.0 [ -1.8, 3.9]","-0.4 [ -1.9, 1.0]","-3.1 [ -4.2, -1.9]","3.0 [ -0.1, 6.2]"


## 강남 결과 7. 폭염 묶음 하나씩 빼고 다시 추정 (하루 전체)

In [17]:
drop_one_run(data, card1, '강남').round(1)

,뺀 묶음,일수,효과%,5%,95%
0,07/07~07/12,6,0.8,-0.4,2.0
1,07/23~08/02,11,0.6,-0.8,2.1
2,08/04~08/05,2,0.9,-0.3,2.1
3,08/10~08/10,1,1.0,-0.2,2.2
4,08/12~08/12,1,1.1,-0.0,2.3
5,08/21~08/22,2,1.1,-0.1,2.4
6,08/24~08/24,1,1.3,0.2,2.5
7,09/05~09/05,1,1.5,0.4,2.6


## 읽을 때 주의할 점
- **휴가철과 섞였을 가능성이 큽니다.** 폭염일이 7월 말~8월 초(휴가 성수기)에 몰려 있어, 월 더미만으로는 휴가 효과를 다 걸러 내지 못합니다. 춘천은 7/22~08/02 폭염 묶음 하나를 빼면 하루 전체 효과가 0 근처로 떨어집니다(결과 7). 그래서 "폭염 날 결제가 늘었다"는 폭염 때문이라고 쓰지 않습니다.
- **가짜 폭염일 검사(placebo)는 하지 않았습니다.** 7~8월 표본에서 폭염이 아닌 날이 춘천 30일, 강남 35일뿐이라, 실제 폭염일 수(29일, 25일)만큼 가짜로 고르면 후보를 거의 다 쓰게 되어 비교가 의미 없습니다.
- **00~05시와 일부 업종은 결제가 아주 적습니다.** 그래서 % 값이 크게 튑니다(예: 춘천 의료·약국, 생활·교육서비스의 00~05시). 해석에서 뺍니다.
- **폭염특보는 7/1부터의 기록만 있습니다.** 6월에 발표되어 7월 초까지 이어진 특보는 알 수 없어, 특보 기준 모형에서는 첫 기록(7/7) 전 날을 뺐습니다.
- **강남 = 서울동남권은 가정입니다** (팀원B 메모 규칙). 서울 기상 지점(108)은 종로구 송월동에 있습니다.
- 거주지가 없어서 **외지인 기준인 호우 분석과 비교 대상이 다릅니다** (이 분석은 전체 결제).
- 상관관계를 보는 분석입니다. "폭염 때문에"가 아니라 "폭염과 관련이 있다"로 씁니다.